In [1]:
# Cell 1: Install
%pip install -q transformers datasets peft accelerate scikit-learn scipy pandas numpy sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [1]:
# Cell 2: Imports
import os, re, inspect
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr

set_seed(42)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

c:\Users\hanib\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [ ]:
# Cell 3: Config
MODEL_NAME = "microsoft/deberta-v3-base"

TRAIN_PATH = r"C:\Users\hanib\Desktop\nlp_final_final_final_final_project\train.csv"
VALID_PATH = r"C:\Users\hanib\Desktop\nlp_final_final_final_final_project\valid.csv"
TEST_PATH  = r"C:\Users\hanib\Desktop\nlp_final_final_final_final_project\test.csv"

OUTPUT_DIR = r"C:\Users\hanib\Desktop\nlp_final_final_final_final_project\deberta_score_model"

MAX_LEN = 512
USE_LORA = True
FREEZE_LOWER_N = 8

In [ ]:
# Cell 4: Load data
train_df = pd.read_csv(TRAIN_PATH)
valid_df = pd.read_csv(VALID_PATH) if os.path.exists(VALID_PATH) else None
test_df = pd.read_csv(TEST_PATH)

INPUT_COLS = ["rubric", "task", "submitted_abstract"]
TARGET_COL = "score"

def extract_score(x):
    """
    Handles:
    3
    3.5
    '3: reject, not good enough'
    """
    if pd.isna(x):
        return np.nan
    match = re.search(r"-?\d+(\.\d+)?", str(x))
    return float(match.group()) if match else np.nan

for df in [train_df, valid_df, test_df]:
    if df is None:
        continue
    for col in INPUT_COLS:
        if col not in df.columns:
            raise ValueError(f"Missing column: {col}")
        df[col] = df[col].fillna("").astype(str)

if TARGET_COL not in train_df.columns:
    raise ValueError("Train file must contain score column")

train_df[TARGET_COL] = train_df[TARGET_COL].apply(extract_score)
train_df = train_df.dropna(subset=[TARGET_COL]).reset_index(drop=True)

if valid_df is not None:
    valid_df[TARGET_COL] = valid_df[TARGET_COL].apply(extract_score)
    valid_df = valid_df.dropna(subset=[TARGET_COL]).reset_index(drop=True)
else:
    valid_df = train_df.sample(frac=0.1, random_state=42)
    train_df = train_df.drop(valid_df.index).reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)

print(train_df.shape, valid_df.shape, test_df.shape)

In [ ]:
# Cell 5: Build DeBERTa input text
def build_text(row):
    return f"""
Rubric:
{row['rubric']}

Task:
{row['task']}

Submitted Abstract:
{row['submitted_abstract']}
""".strip()

train_df["text"] = train_df.apply(build_text, axis=1)
valid_df["text"] = valid_df.apply(build_text, axis=1)
test_df["text"] = test_df.apply(build_text, axis=1)

train_ds = Dataset.from_pandas(train_df[["text", TARGET_COL]], preserve_index=False)
valid_ds = Dataset.from_pandas(valid_df[["text", TARGET_COL]], preserve_index=False)

test_cols = ["text"]
if TARGET_COL in test_df.columns:
    test_df[TARGET_COL] = test_df[TARGET_COL].apply(extract_score)
    test_cols.append(TARGET_COL)

test_ds = Dataset.from_pandas(test_df[test_cols], preserve_index=False)

In [ ]:
# Cell 6: Tokenization
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    result = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN
    )
    if TARGET_COL in batch:
        result["labels"] = [float(x) for x in batch[TARGET_COL]]
    return result

train_ds = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
valid_ds = valid_ds.map(tokenize, batched=True, remove_columns=valid_ds.column_names)
test_tok = test_ds.map(tokenize, batched=True, remove_columns=test_ds.column_names)

collator = DataCollatorWithPadding(tokenizer)

In [ ]:
# Cell 7: Model + LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    problem_type="regression"
)

if USE_LORA:
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=16,
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=["query_proj", "value_proj"],
        bias="none"
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
else:
    for i, layer in enumerate(model.deberta.encoder.layer):
        if i < FREEZE_LOWER_N:
            for p in layer.parameters():
                p.requires_grad = False

In [ ]:
# Cell 8: Metrics
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.squeeze(preds)
    labels = np.squeeze(labels)

    rmse = mean_squared_error(labels, preds, squared=False)
    mae = mean_absolute_error(labels, preds)
    r2 = r2_score(labels, preds)

    try:
        pearson = pearsonr(labels, preds)[0]
    except:
        pearson = 0.0

    try:
        spearman = spearmanr(labels, preds)[0]
    except:
        spearman = 0.0

    return {
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "pearson": pearson,
        "spearman": spearman
    }

In [ ]:
# Cell 9: TrainingArguments compatible with your transformers version
args_dict = dict(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-4 if USE_LORA else 2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="rmse",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

sig = inspect.signature(TrainingArguments.__init__)
if "eval_strategy" in sig.parameters:
    args_dict["eval_strategy"] = "epoch"
else:
    args_dict["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**args_dict)

In [ ]:
# Cell 10: Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics
)

trainer.train()

print(trainer.evaluate())

In [ ]:
# Cell 11: Inference on test without score
pred = trainer.predict(test_tok)
pred_scores = np.squeeze(pred.predictions)

result_df = test_df.copy()
result_df["predicted_score"] = pred_scores

# Optional: clip if your score range is 1 to 5
result_df["predicted_score_clipped"] = result_df["predicted_score"].clip(1, 5)

save_path = r"C:\Users\hanib\Desktop\nlp_final_final_final_final_project\test_predictions.csv"
result_df.to_csv(save_path, index=False)

print("Saved:", save_path)
result_df.head()

In [ ]:
# Cell 12: Evaluate test only if score exists
if "score" in result_df.columns:
    mask = result_df["score"].notna()
    y_true = result_df.loc[mask, "score"].values
    y_pred = result_df.loc[mask, "predicted_score"].values

    print("Test RMSE:", mean_squared_error(y_true, y_pred, squared=False))
    print("Test MAE:", mean_absolute_error(y_true, y_pred))
    print("Test R2:", r2_score(y_true, y_pred))
    print("Pearson:", pearsonr(y_true, y_pred)[0])
    print("Spearman:", spearmanr(y_true, y_pred)[0])
else:
    print("No score column in test. Only predictions were generated.")